## TOPAZ: TOpologically-based Parameter inference for Agent-based model optimiZation

This code has three big sections with two additional optionals steps: the simulation or topological data analysis (TDA) step, the parameter estimation step using approximate Bayesian computation (ABC), the (optional) approximate approximate Bayesian computation (AABC), the (optional) statistical verification step using PERMANOVA, Energy Distance, and/or MMD tests, and the Bayesian information criterion step (BIC).

<img src="TOPAZ_figure.png" alt="TOPAZ workflow" width="800"/>


### Overview

This general notebook provides model-agnostic scaffolding for running the TOPAZ pipeline with your own agent-based model (ABM). Steps that depend on the ABM are intentionally left as adapter points; reusable loss, posterior, AABC, statistical-verification, and BIC operations are included or connected to the general modules.

The original ABC helper for posterior-density slices is designed around three parameters. The AABC cells added here accept an arbitrary number of parameters, provided every reference parameter vector has the same length and every CROCKER array has the same shape.

#### Included

- Calculating losses between the ground-truth and sampled CROCKER arrays
- Estimating ABC medians and posterior-density plots
- Generating extra, approximate samples with AABC
- Comparing posterior-predictive CROCKER distributions using PERMANOVA, energy distance, and MMD
- Calculating CROCKER differences, SSE, BIC, and AIC

#### Supply for your ABM

- A simulation adapter that accepts one parameter vector and returns a trajectory
- A TDA adapter that converts a trajectory to a consistently shaped CROCKER array
- Prior bounds and initial conditions
- Two posterior-predictive sample groups when comparing competing models

Run the sections in numerical order. Expensive sampling cells are disabled by default; review paths and set their run flags explicitly.


In [ ]:
# necessary imports needed throughout
from IPython.display import Image,display, 
from pathlib import Path
import numpy as np

### Step 1: Topological Data Analysis

#### 1.1 Simulate the ABM simulation

In [ ]:
'''Run ABM simulation

Recommended Inputs:
    Parameters - Par1, Par2, Par3, etc. 
    T0 - Initial time of simulation
    TF - End time of simulation
    DT - How often to make a new frame of data
    num_agents - number of agents to be in the simulation

Recommended Requirements:
    ic_vec.npy - vector of initial conditions
    
Recommended Outputs: 
    df_TDA.pkl - the resulting TDA dataframe over the simulation
    TDA_simulation.gif - gif of chosen simulation
    
'''

# upload simulation results
tda_sims = './tda_simulations' #update 

#indices for parameter values chosen to run the TDA simulation (needed in step 2d)
Par1_idx,Par2_idx,Par3_idx = [] #update 

# display simulation gif - may upload your own example for viewing
# general_gif_name = f"./TDA_simulation.gif"
# display(HTML(f'<img src="{general_gif_name}" />'))

This simulation will represent our "ground truth" baseline moving forward.

#### 1.2 Run TDA to get the Betti-0 and Betti-1 crocker matrices for your (Par1, Par2, Par3+) combination

In [ ]:
'''Compute crockers for specific Betti numbers given a trajectory dataframe.

Recommended Inputs:
    df_TDA.pkl - the resulting TDA dataframe over the simulation
    
Recommended Output: 
    TDA_crocker_angles.npy - the array of crocker angles that form the crocker plot
    
'''

# upload or calculate the crocker plots for each TDA simulation above
tda_crocker_angles_path = './tda_crocker_angles.npy' #update #this is the simulation for one parameter combination, if more wanted, will need to loop over the combinations


# display crocker plot - may upload your own example for viewing
# tda_crocker_name = f"./tda_crocker_plot.png"
# display(Image(filename=tda_crocker_name, width=600))

### Step 2: Approximate Bayesian Computation

#### 2.1 Generate random samples

The goal is to run a large amount of ABC simulations. However, this is a very time and space consuming process. A sample of 30 random simulations has been pre-run but for more thourough analysis, closer to 10,000 samples is recommended.

In [ ]:
'''Run ABM simulation for random values of Par1,Par2,Par3 over a specified time period

Recommended Inputs:
    T0 - Initial time of simulation
    TF - End time of simulation
    DT - How often to make a new frame of data
    num_agents - number of cell agents to be in the simulation
    
Recommended Requirements:
    ic_vec.npy - vector of initial conditions
    
Recommended Output: 
    df_ABC.pkl - the resulting dataframe over the simulation with random Par1, Par2, and Par3 values
    pars.npy - file of parameters used in random simulation
    random_ABC_simulation.gif - gif of random Par1,Par2,Par3 simulation
    
'''

# upload x simulation results or run x number of random simulations
abc_sims = './abc_simulations' #update

# display simulation gif - may upload your own example for viewing
# display(HTML('<img src="./random_ABC_simulation.gif">'))

#### 2.2 Calculate crocker plots for random samples 

This step is also very time consuming and similarly has been uploaded with an option to calculate the crocker plot for the single simulation ran in the previous step. 

In [ ]:
'''Compute crockers for specific Betti numbers given a trajectory dataframe for random Par1,Par2,Par3 values

Recommended Inputs:
    df_ABC.pkl - the resulting dataframe over the simulation with random Par1, Par2, and Par3 values
    pars.npy - file of parameters used in random simulation
    
Recommended Output: 
    ABC_crocker_angles.npy - crocker values of random Par1,Par2,Par3 simulation (one for each random simulation)
    
'''

# path to the saved crocker_angles.npy files
abc_crocker_angles_and_pars_path = './your_sample_path' #update

#### 2.3 Calculate samples losses for each crocker plot 

Now we will begin comparing our ABC results to our ground truth simulation from step 1. 

In [ ]:
# calculate the sample loss and distance between our ground truth results and ABC results 

from Modules_General.ABC3_compute_losses import compute_losses

'''Compute sample losses between ground truth and random ABC simulations and crocker plots 

Inputs:
    num_samples - number of samples contained in samples_path
    tda_crocker_angles_path - crocker values of chosen Par1,Par2,Par3 simulation from 1b
    abc_crocker_angles_and_pars_path - path to the random samples from 2b
    sample_losses_angles_path - desired path for sample losses angles to be saved to
    
Output: 
    sample_losses_angles.npy - file of sample losses calculated for the random samples 
    
'''
num_samples = 30 #update
sample_losses_angles_path = './sample_losses_angles.npy' #update

sample_losses = compute_losses(num_samples, tda_crocker_angles_path, abc_crocker_angles_and_pars_path, sample_losses_angles_path)

#### 2.4 Calculate ABC medians and posterior density plots 

From the sample losses, we can calculate the median values for ABC and create corresponding posterior density plots. 

In [ ]:
# calculate medians 
from Modules_General.ABC4_medians import compute_medians_and_densities

'''Create posterior density plots for calculated sample losses and median ABC estimated values for Par1,Par2,Par3

Inputs:
    Par1_idx, Par2_idx, Par3_idx - indices to parameter values chosen for TDA simulation
    sample_losses_angles_path - path to sample losses angles generated in 2c
    abc_posterior_densities_path - desired path for posterior densities to be saved to
    median_path - desired path for median values to be saved to
    
Output: 
    ABC_posterior_densities - folder containing 2D ABC posterior slices (1 for each Par3 value)
    medians - ABC estimated median values for Par1,Par2,Par3

'''
#path for the ABC posterior density plots to be saved within
abc_posterior_densities_path = './ABC_posterior_densities' #update
median_path = './medians.npy'

#Par1_idx,Par2_idx,Par3_idx refer to ground truth indices for Par1, Par2, and Par3
medians = compute_medians_and_densities(Par1_idx,Par2_idx,Par3_idx,sample_losses_angles_path,abc_posterior_densities_path,median_path)

print('Medians: ' + str(medians))

#output posterior density plots 
html = "" 
for Par3_slice in range(11): #update 
    post_slices_name = f"/ABC_posterior_densities/posterior_density_slice_at_w{str(Par3_slice).zfill(2)}.png"
    html += f'<img src="{post_slices_name}" width="200" style="margin-right:10px;" />'

display(HTML(html))

#### 2.5 Run the ABC model simulations

Next we will run the ABC medians through our ABM simulation. 

In [ ]:
'''Run ABM simulation for ABC median estimated values of Par1, Par2, and Par3 over a specified time period

Recommended Inputs:
    Medians - median values for parameters at which to run the ABC simulation on 
    T0 - Initial time of simulation
    TF - End time of simulation
    DT - How often to make a new frame of data
    num_agents - number of agents to be in the simulation

Recommended Requirements:
    ic_vec.npy - vector of initial conditions
    
Recommended Outputs: 
    df_median.pkl -  the resulting dataframe over the simulation with median Par1, Par2, and Par3 values
    ABC_median_simulation.gif - gif of median simulation
    
'''

#run ABC simulation for median values 
abc_median_sim = './abc_median_simulation' #update 

# display simulation gif
# ABC_med_gif_name = f"/ABC_median_simulation.gif"
# display(HTML(f'<img src="{ABC_med_gif_name}" />'))

#### 2.6 Calculate the ABC crocker plots 

From the simulation, we can create an ABC crocker plot. 

In [ ]:
'''Compute crockers for specific Betti numbers given a trajectory dataframe for ABC median estimated Par1,Par2,Par3 values

Recommended Inputs:
    df_median.pkl - the resulting dataframe over the simulation with median Par1, Par2, and Par3 values
    
Recommended Output: 
    median_crocker_angles.npy - crocker values of median Par1,Par2,Par3 simulation
    true_crocker.png - Ground truth crocker plot 
    ABC_crocker.png - ABC median crocker plot 
    
'''

# path to the saved median_crocker_angles.npy files
median_crocker_angles_path = './median_crocker_angles.npy' 

# display true crocker plot 
# tda_crocker_name = f"./tda_crocker_plot.png"
# display(Image(filename=tda_crocker_name, width=600))

# display ABC crocker plot 
# abc_median_crocker_name = f"./median_crocker_plot.png"
# display(Image(filename=abc_median_crocker_name, width=600))

#### 2.7 Calculate the differences between the TDA crocker and ABC median crocker

Now that we have the TDA crocker and ABC median crocker, we can calculate the difference between then which will help lead us to the SSE. 

In [ ]:
from Modules_General.ABC7_crocker_difference import compute_crocker_difference

'''Compute the difference between the TDA crocker and ABC median crocker

Recommended Inputs:
    TDA_crocker_angles.npy - the array of crocker angles that form the crocker plot
    median_crocker_angles.npy - crocker values of median Par1,Par2,Par3 simulation
    
Recommended Output: 
    crocker_differences.npy - calculated difference between the two crocker plots 
    
'''

#path for crocker differences to be saved to
crocker_difference_path = './crocker_differences.npy'

#compute the crocker difference
compute_crocker_difference(tda_crocker_angles_path,median_crocker_angles_path,crocker_difference_path)


### Step 2a (optional): Approximate Approximate Bayesian Computation

AABC can cheaply expand a reference ABC library by interpolating the summary statistics of nearby parameter vectors. It is an approximation: the generated CROCKER arrays are not new ABM simulations. 


#### 2a.1 Choose the AABC domain

Optionally shrink the prior bounds using the ABC posterior. If you shrink them, first generate enough ordinary ABC reference samples in the new domain. Every AABC proposal should lie inside a region supported by the reference library.


In [ ]:
'''Go through posterior density plots and decide on a smaller, denser grid to run the AABC step on. 
    
'''

# One [lower, upper] row per model parameter, in the same order as pars.npy.
parameter_names = ["Par1", "Par2", "Par3"]  # update
parameter_bounds = np.array([
    [0.0, 1.0],  # Par1: update
    [0.0, 1.0],  # Par2: update
    [0.0, 1.0],  # Par3: update
], dtype=float)

if parameter_bounds.shape != (len(parameter_names), 2):
    raise ValueError("parameter_bounds must contain one [lower, upper] row per parameter")
if np.any(parameter_bounds[:, 0] >= parameter_bounds[:, 1]):
    raise ValueError("Every lower parameter bound must be smaller than its upper bound")


#### 2a.2 Extract reference parameters and flatten CROCKER arrays

Each reference run directory must contain `pars.npy` and `crocker_angles.npy`. The loader validates parameter dimensions and CROCKER shapes before saving the stacked reference library.


In [ ]:
from Modules_General.AABC1_get_params_n_crockers import get_params_n_crockers_aabc


'''Extract the parameters used in ABC samples and flatten Crocker matrices generated in ABC samples

Inputs:
    samples_path - path to ABC samples to use in AABC (whether original samples or smaller grid samples)
    parameters - an array of your model parameters

Requirements:
    crocker_angles.npy - crocker values of random sampled C,L,W simulations
    pars.npy - file of parameters used in random sampled C,L,W simulations
        
Output: 
    all_params.npy - All of the parameters used in the orignal samples 
    all_crockers_flattened.npy - All of the Crockers generated in the original samples flattened
    
'''

aabc_samples_path = '/aabc_samples' #update
parameters = [Par1, Par2, Par3]

[params, crockers] = get_params_n_crockers_aabc(aabc_samples_path, parameters)

aabc_reference_path = Path("./aabc_reference")  # update if desired
params, crockers, crocker_shape = get_params_n_crockers_aabc(
    abc_crocker_angles_and_pars_path,
    aabc_reference_path,
    num_parameters=len(parameter_names),
)
print(f"Loaded {len(params)} reference runs; CROCKER shape: {crocker_shape}")


#### 2a.3 Generate AABC samples

For each proposed parameter vector, the Epanechnikov kernel weights its nearest reference neighbors. A Dirichlet draw then mixes their flattened CROCKER arrays. `k` must be smaller than the number of reference samples.


In [ ]:
from Modules_General.AABC2_run_samples import run_samples_aabc

'''Generate new CROCKER samples using the AABC methodology.

Inputs:
    aabc_reference_path - path to the reference parameters and flattened CROCKER arrays
    aabc_samples_path - folder in which to save the generated AABC samples
    parameter_bounds - one [lower, upper] row for each model parameter
    num_aabc_samples - number of AABC samples to generate
    k_neighbors - number of nearest reference samples used for interpolation
    seed - random seed used for reproducibility

Requirements:
    all_params.npy - parameter vectors extracted from the original ABC samples
    all_crockers_flattened.npy - flattened CROCKER arrays from the original ABC samples
    crocker_shape.npy - original shape of the CROCKER arrays

Outputs:
    For each AABC sample:
        theta_star.npy - sampled parameter vector
        crocker_angles.npy - generated AABC CROCKER array
    sampled_params.npy - parameter vectors for all generated AABC samples
'''

num_aabc_samples = 120  # update
k_neighbors = 5  # update
aabc_samples_path = f'./sample_aabc_{num_aabc_samples}'

sampled_params = run_samples_aabc(
    aabc_reference_path,
    aabc_samples_path,
    parameter_bounds,
    num_aabc_samples,
    k=k_neighbors,
    seed=2026,
)

print(f'Generated {len(sampled_params)} AABC samples in {aabc_samples_path}')


#### 2a.4 ABC+AABC Sample Loss Calculation

Similar to step 2.3 above, calculate the sample losses for the original ABC samples and the newly generated AABC samples together.


In [ ]:
from Modules_General.AABC3_compute_losses import compute_losses_aabc

'''Compute losses between the ground-truth CROCKER and the ABC+AABC CROCKER samples.

Inputs:
    tda_crocker_angles_path - path to the ground-truth CROCKER array from step 1.2
    abc_crocker_angles_and_pars_path - path to the original ABC run folders
    aabc_samples_path - path to the AABC run folders generated in step 2a.3
    sample_losses_angles_aabc_path - path at which to save the combined losses

Requirements:
    TDA_crocker_angles.npy - ground-truth CROCKER array
    For every ABC run: pars.npy and crocker_angles.npy
    For every AABC run: theta_star.npy and crocker_angles.npy

Outputs:
    sample_losses_angles_aabc.npz - methods, parameter vectors, and losses for all ABC+AABC samples
'''

sample_losses_angles_aabc_path = './sample_losses_angles_aabc.npy'  # update

methods_aabc, parameters_aabc, sample_losses_aabc = compute_losses_aabc(
    tda_crocker_angles_path,
    abc_crocker_angles_and_pars_path,
    aabc_samples_path,
    sample_losses_angles_aabc_path,
)


#### 2a.5 Repeat ABC Steps 2.4, 2.5, and 2.6

Repeat the posterior estimation, median simulation, and CROCKER calculation using the combined ABC+AABC losses. Save each result with `aabc` in its filename so the original ABC results are not overwritten.


##### 2.4 repeated: Calculate AABC medians and posterior-density plots


In [ ]:
from Modules_General.AABC4_medians import compute_medians_and_densities_aabc

'''Create posterior-density plots and estimate parameter medians using ABC+AABC losses.

Inputs:
    sample_losses_angles_aabc_path - path to the combined losses from step 2a.4
    aabc_posterior_densities_path - folder in which to save posterior-density plots
    medians_aabc_path - path at which to save the AABC-estimated medians

Requirements:
    sample_losses_angles_aabc.npz - aligned parameter vectors and ABC+AABC losses
    Parameter grid information used by your posterior-estimation script

Outputs:
    AABC_posterior_densities - posterior-density plots
    medians_aabc.npy - AABC-estimated median value for each model parameter
'''

aabc_posterior_densities_path = './AABC_posterior_densities'  # update
medians_aabc_path = './medians_aabc.npy'  # update

medians_aabc = compute_medians_and_densities_aabc(
    sample_losses_angles_aabc_path,
    aabc_posterior_densities_path,
    medians_aabc_path,
)

print('AABC medians: ' + str(medians_aabc))


##### 2.5 repeated: Run the ABM using the AABC-estimated medians


In [ ]:
from Modules_General.Modules_you_replace.AABC5_run_ABC_sim import run_ABC_sim_aabc

'''Run your ABM using the AABC-estimated median parameter values.

Inputs:
    medians_aabc_path - path to the AABC-estimated medians
    T0 - initial simulation time
    TF - final simulation time
    DT - interval at which to save simulation frames
    num_agents - number of agents in the simulation

Requirements:
    medians_aabc.npy - AABC-estimated median parameter values
    Any initial-condition files required by your ABM

Outputs:
    df_AABC.pkl - trajectory dataframe generated at the AABC medians
    AABC_median_simulation.gif - optional visualization of the simulation
'''

# Update the arguments to match your ABM adapter.
run_ABC_sim_aabc(medians_aabc_path, T0, TF, DT, num_agents)


##### 2.6 repeated: Calculate the AABC median CROCKER array


In [ ]:
from Modules_General.Modules_you_replace.AABC6_run_ABC_crocker import run_ABC_crocker_aabc

'''Compute and compare the ground-truth and AABC-median CROCKER arrays.

Inputs:
    tda_crocker_angles_path - path to the ground-truth CROCKER array
    df_aabc_path - path to the trajectory generated at the AABC medians
    aabc_crocker_angles_path - path at which to save the AABC CROCKER array
    crocker_difference_aabc_path - path at which to save the CROCKER differences

Requirements:
    TDA_crocker_angles.npy - ground-truth CROCKER array
    df_AABC.pkl - trajectory generated using the AABC medians

Outputs:
    AABC_crocker_angles.npy - CROCKER array for the AABC-median simulation
    crocker_differences_aabc.npy - difference between ground-truth and AABC CROCKER arrays
    AABC_crocker.png - optional CROCKER visualization
'''

df_aabc_path = './df_AABC.pkl'  # update
aabc_crocker_angles_path = './AABC_crocker_angles.npy'  # update
crocker_difference_aabc_path = './crocker_differences_aabc.npy'  # update

run_ABC_crocker_aabc(
    tda_crocker_angles_path,
    df_aabc_path,
    aabc_crocker_angles_path,
    crocker_difference_aabc_path,
)


### Step 2b (Optional): Statistical Verification

These tests assess whether the posterior-predictive CROCKER summaries generated by two models are statistically distinguishable. The three tests used are PERMANOVA, energy distance, and maximum mean discrepancy (MMD).


#### 2b.1 Generate samples from the posterior densities

Run each competing ABM at parameter values drawn from its posterior. This step can take several hours, so the function call is commented out by default.


In [ ]:
from Modules_General.Modules_you_replace.STAT1_run_posterior_samples import run_posterior_samples

'''Generate independent posterior-predictive simulations for one model.

Inputs:
    model_name - label for the model being simulated
    posterior - posterior distribution or accepted parameter samples
    num_post_samples - number of posterior-predictive simulations to run
    posterior_samples_path - folder in which to save the simulations
    simulate_abm - function that runs the model for one parameter vector
    rng - NumPy random-number generator

Requirements:
    Posterior results generated during parameter inference
    Any initial-condition files required by the ABM
    A completed model-specific STAT1 adapter in Modules_you_replace

Outputs:
    For every run: trajectory file and pars.npy
    Optional simulation visualization files
'''

num_post_samples = 50  # update
model_a_posterior_runs = './posterior_samples/model_a'  # update
model_b_posterior_runs = './posterior_samples/model_b'  # update

# Uncomment after completing the model-specific adapter.
# run_posterior_samples('model_a', posterior_a, num_post_samples,
#                       model_a_posterior_runs, simulate_model_a, np.random.default_rng(1))
# run_posterior_samples('model_b', posterior_b, num_post_samples,
#                       model_b_posterior_runs, simulate_model_b, np.random.default_rng(2))


#### 2b.2 Generate CROCKER arrays for the posterior-predictive samples


In [ ]:
from Modules_General.Modules_you_replace.STAT2_run_posterior_crockers import run_posterior_crockers

'''Calculate a CROCKER array for every posterior-predictive simulation.

Inputs:
    posterior_samples_path - folder containing the posterior-predictive runs
    trajectory_to_crocker - function that converts one trajectory to a CROCKER array

Requirements:
    A trajectory file in every posterior-predictive run folder
    The same filtration, Betti numbers, time grid, and normalization used during inference
    A completed model-specific STAT2 adapter in Modules_you_replace

Outputs:
    crocker_angles.npy - posterior-predictive CROCKER array in every run folder
'''

# Uncomment after completing the model-specific adapter.
# run_posterior_crockers(model_a_posterior_runs, trajectory_to_crocker_a)
# run_posterior_crockers(model_b_posterior_runs, trajectory_to_crocker_b)


#### 2b.3 Run Statistical Verification

A small p-value indicates evidence that the posterior-predictive distributions differ; it does not identify which model is better.


In [ ]:
from Modules_General.STAT3_statistical_verification import statistical_verification

'''Compare the posterior-predictive CROCKER distributions from two models.

Inputs:
    model_a_posterior_runs - path to the first model's posterior-predictive runs
    model_b_posterior_runs - path to the second model's posterior-predictive runs
    stats_results_path - path at which to save the statistical-test results
    permutations - number of permutations used to calculate p-values
    seed - random seed used for reproducibility

Requirements:
    crocker_angles.npy in every run folder for both models
    Identical CROCKER array shapes and preprocessing for the two groups
    pandas, scikit-learn, scikit-bio, and hyppo

Outputs:
    stats_results.csv - PERMANOVA, energy-distance, and MMD statistics and p-values
'''

stats_results_path = './Statistical_Verification_Files/stats_results.csv'  # update

stats_results = statistical_verification(
    model_a_posterior_runs,
    model_b_posterior_runs,
    stats_results_path,
    permutations=999,
    seed=2026,
)

display(stats_results)


### Step 3: Bayesian Information Criterion


#### 3.1 Calculate the BIC score


Calculate the BIC score from the SSE between the ground-truth and estimated-model CROCKER arrays.


In [ ]:
from Modules_General.BIC_calc_bic import calc_bic

'''Calculate the BIC score for the fitted model.

Inputs:
    num_parameters - number of estimated model parameters
    num_data_pts - number of values compared between the two CROCKER arrays
    crocker_difference_path - path to the ABC or AABC CROCKER-difference file
    bic_path - path at which to save the BIC results

Requirements:
    crocker_differences.npy or crocker_differences_aabc.npy

Outputs:
    BIC score - information-criterion score for the fitted model
    SSE - sum of squared errors between the ground-truth and estimated CROCKER arrays
    bic_results.npy - saved BIC and SSE results
'''

num_parameters = len(parameter_names)  # update if needed
num_data_pts = 40000  # update
bic_path = './bic_results.npy'  # update

# Use crocker_difference_aabc_path for the AABC result, or crocker_difference_path for ABC.
bic_score, sse_score = calc_bic(
    num_parameters,
    num_data_pts,
    crocker_difference_aabc_path,
    bic_path,
)

print('BIC Score: ' + str(bic_score) + ' SSE: ' + str(sse_score))
